 results from both results and sprint table , only facts table , not dimensions

In [0]:
%run ../common/environmental_config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
df_results = (spark.table(f"{catalog_name}.{silver_schema}.results")
              .withColumn('session_type' , F.lit('race'))
              .drop("race_name", "race_date", "ingestion_timestamp", "source_file"))
df_sprints = (spark.table(f"{catalog_name}.{silver_schema}.sprints")
              .withColumn('session_type' , F.lit('sprint'))
              .drop("race_name", "race_date", "ingestion_timestamp", "source_file"))

In [0]:
results_sprints_df = df_results.unionByName(df_sprints)

In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("finish_position") == 1)
        .withColumn("is_podium", F.col("finish_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

In [0]:
(
    fact_session_results_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)